# FITILA — entraînement du traducteur FR ↔ Baatɔnum (ByT5)
Kaggle : *Accelerator* = GPU T4 x2 ou P100, *Internet* = On. Ajoutez le dataset `dunya-translate-kit` (le zip fourni).
Optionnel : secret Kaggle `HF_TOKEN` pour repartir des poids v03 (sinon `google/byt5-small`).

In [ ]:
# 1) Trouver le kit, où que Kaggle l'ait monté (/kaggle/input/<slug>/ ou /kaggle/input/datasets/<user>/<slug>/...)
import os, glob, shutil
hits = glob.glob('/kaggle/input/**/scripts/dunya_translate/train_byt5.py', recursive=True)
assert hits, 'Kit introuvable : ajoutez le dataset dunya-translate-kit (panneau Input > Add Input).'
src = hits[0].rsplit('/scripts/', 1)[0]
ROOT = '/kaggle/working/kit'
shutil.rmtree(ROOT, ignore_errors=True)
shutil.copytree(src, ROOT)  # /kaggle/input est en lecture seule : on travaille sur une copie
print('kit :', src, '->', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# 2) Dépendances, GPU, modèle de départ
!pip -q install transformers sentencepiece accelerate sacrebleu
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (activez Accelerator = GPU T4 x2)')
os.chdir(ROOT + '/scripts/dunya_translate')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
MODEL = 'zimesongbian/modele_byt5_bariba_expert_api_v03_improve' if os.environ.get('HF_TOKEN') else 'google/byt5-small'
LR = '1e-4' if os.environ.get('HF_TOKEN') else '3e-4'
print('modèle de départ :', MODEL)

In [ ]:
!python -m unittest test_pipeline 2>&1 | tail -3
!python make_training_data.py

## 1. Fine-tuning supervisé (indices mémoire + glossaire, Apprendre multitâche)

In [ ]:
!python train_byt5.py --model $MODEL --out /kaggle/working/v04 --epochs 6 --lr $LR --batch 8 --accum 4

## 2. Renforcement : exemples difficiles puis préférences (DPO)

In [ ]:
!python mine_hard_examples.py --model /kaggle/working/v04 --samples 6 --limit 4000
!python dpo.py --model /kaggle/working/v04 --out /kaggle/working/v04-dpo --epochs 1 --batch 8

## 3. Mesure sur le jeu de test (mémoire = train seulement)

In [ ]:
!python eval_translation.py --backend hf --model /kaggle/working/v04 --rag --out /kaggle/working/eval_v04_rag.json | head -40
!python eval_translation.py --backend hf --model /kaggle/working/v04-dpo --rag --out /kaggle/working/eval_v04_dpo_rag.json | head -40

## 4. Export ONNX hors ligne + archive à télécharger
Comparer chrF++ avec la ligne de base v03 (26,0 FR→BA / 23,6 BA→FR avec autocorrection ; 26,6 / 26,9 brut).

In [ ]:
!pip -q install onnx onnxruntime
!python ../dunya_byt5/export_byt5_onnx.py --model /kaggle/working/v04-dpo --out /kaggle/working/onnx --sample 'fr-ba: Bonjour' --sample 'ba-fr: A kua'
!cd /kaggle/working && zip -qr dunya-translate-result.zip v04-dpo onnx eval_*.json && ls -la dunya-translate-result.zip